### Game Level 3 Apps Deployment

Deploys two Databricks Apps for Level 3 of Casper's Kitchen Rescue:
1. **City Operations Dashboard** — traffic, weather, events per city
2. **Kitchen Operations Dashboard** — staffing, prep time per kitchen

In [ ]:
%pip install databricks-sdk --upgrade

In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")

In [ ]:
import sys
sys.path.append('../utils')
from uc_state import add

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

review_warehouse_name = f"{CATALOG}-game-warehouse"

warehouse_id = None
for wh in w.warehouses.list():
    if wh.name and wh.name in (review_warehouse_name, f"{CATALOG}-warehouse"):
        warehouse_id = wh.id
        print(f"\u267b\ufe0f Reusing existing warehouse '{wh.name}' (id={wh.id})")
        break

if not warehouse_id:
    warehouse = w.warehouses.create(
        name=review_warehouse_name,
        cluster_size="2X-Small",
        max_num_clusters=1,
        min_num_clusters=1,
        enable_serverless_compute=True
    ).result()
    warehouse_id = warehouse.id
    add(CATALOG, "warehouses", warehouse)
    print(f"\u2705 Created warehouse (id={warehouse_id})")
else:
    print(f"\u2705 Using warehouse_id={warehouse_id}")

In [ ]:
import os, time

app_yaml = f"""command:
  - uvicorn
  - app.main:app
env:
  - name: DATABRICKS_WAREHOUSE_ID
    value: '{warehouse_id}'
  - name: DATABRICKS_CATALOG
    value: '{CATALOG}'
"""

for app_dir in ["../apps/order-review", "../apps/kitchen-ops"]:
    yaml_path = os.path.join(app_dir, "app.yaml")
    if os.path.exists(yaml_path):
        os.remove(yaml_path)
    time.sleep(1)
    with open(yaml_path, "w") as f:
        f.write(app_yaml)
    print(f"\u2705 Wrote {yaml_path}")

In [ ]:
from databricks.sdk.service.apps import App, AppResource, AppResourceSqlWarehouse, AppResourceSqlWarehouseSqlWarehousePermission
import os

apps_config = [
    {"name": f"{CATALOG}-city-ops", "dir": "../apps/order-review", "label": "City Operations"},
    {"name": f"{CATALOG}-kitchen-ops", "dir": "../apps/kitchen-ops", "label": "Kitchen Operations"},
]

app_results = {}
for cfg in apps_config:
    src = os.path.abspath(cfg["dir"])
    existing = None
    try:
        existing = w.apps.get(cfg["name"])
        print(f"\u267b\ufe0f {cfg['label']} app '{cfg['name']}' exists, will redeploy")
    except Exception:
        pass

    if not existing:
        pending = w.apps.create(
            App(
                name=cfg["name"],
                default_source_code_path=src,
                resources=[
                    AppResource(
                        name="sql-warehouse",
                        sql_warehouse=AppResourceSqlWarehouse(
                            id=warehouse_id,
                            permission=AppResourceSqlWarehouseSqlWarehousePermission.CAN_USE
                        )
                    )
                ]
            )
        )
        app_status = pending.result()
        add(CATALOG, "apps", app_status)
        print(f"\u2705 Created {cfg['label']} app: {app_status.name}")
    else:
        app_status = existing
        print(f"\u2705 Using existing {cfg['label']} app: {app_status.name}")

    app_results[cfg["name"]] = {"status": app_status, "src": src, "label": cfg["label"]}

In [ ]:
print(f"\u2705 Both apps ready: {list(app_results.keys())}")

In [ ]:
from databricks.sdk.service import catalog

for app_info in app_results.values():
    principal = app_info["status"].id
    for schema_name in ["game", "lakeflow", "simulator"]:
        try:
            w.grants.update(
                full_name=f"{CATALOG}.{schema_name}",
                securable_type="SCHEMA",
                changes=[
                    catalog.PermissionsChange(
                        add=[catalog.Privilege.USE_SCHEMA, catalog.Privilege.SELECT],
                        principal=principal
                    )
                ]
            )
        except Exception as e:
            print(f"\u26a0\ufe0f Grant on {schema_name} for {app_info['label']}: {e}")

    try:
        w.grants.update(
            full_name=f"{CATALOG}",
            securable_type="CATALOG",
            changes=[
                catalog.PermissionsChange(
                    add=[catalog.Privilege.USE_CATALOG],
                    principal=principal
                )
            ]
        )
    except Exception as e:
        print(f"\u26a0\ufe0f Catalog grant for {app_info['label']}: {e}")

    print(f"\u2705 Permissions granted for {app_info['label']}")

In [ ]:
from databricks.sdk.service.apps import AppDeployment

app_urls = {}
for app_name_key, app_info in app_results.items():
    deployment = w.apps.deploy(
        app_name=app_info["status"].name,
        app_deployment=AppDeployment(source_code_path=app_info["src"])
    )
    deployment.result()
    url = f"{w.config.host}/apps/{app_info['status'].name}"
    app_urls[app_name_key] = url
    print(f"\u2705 Deployed {app_info['label']}: {url}")

In [ ]:
city_ops_url = app_urls.get(f"{CATALOG}-city-ops", "")
kitchen_ops_url = app_urls.get(f"{CATALOG}-kitchen-ops", "")

for key, val in [("city_ops_url", city_ops_url), ("kitchen_ops_url", kitchen_ops_url)]:
    spark.sql(f"""
    MERGE INTO {CATALOG}.game.config AS target
    USING (SELECT '{key}' AS config_key, '{val}' AS config_value) AS source
    ON target.config_key = source.config_key
    WHEN MATCHED THEN UPDATE SET config_value = source.config_value
    WHEN NOT MATCHED THEN INSERT (config_key, config_value) VALUES (source.config_key, source.config_value)
    """)

print(f"\n\U0001f50d Level 3 Apps are live!")
print(f"   City Operations: {city_ops_url}")
print(f"   Kitchen Operations: {kitchen_ops_url}")